# 10e Confirm and Report Frozen Concepts

Run only after concept definitions and selection are frozen on development pairs. This notebook evaluates the final catalog on confirmation pairs; do not use these results to revise concepts or choose additional candidates.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from amr.pipeline_config import cfg, load_pipeline_config
from amr.pipelines.concept_notebook_workflows import load_concept_catalog_context, confirm_frozen_concepts
from amr.pipeline_config import cfg, load_pipeline_config
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
FIT_START_YEAR = int(cfg(PIPELINE_CONFIG, "si.fit_start_year", 2000))
FORECAST_ORIGIN_END_YEAR = int(cfg(PIPELINE_CONFIG, "si.fit_end_year", 2023))
LAG_LENGTH = 3
FORECAST_HORIZONS = (1, 2, 3)
MIN_PRIOR_OBSERVATIONS = 10
MIN_ATTRIBUTION_ORIGINS = 5


## Configure the frozen model population

This configured run confirms the nine-seed catalog on held-out pairs after development evaluation. The default report evaluates each catalog concept against the SI-only baseline. Populate `FROZEN_POPULATION` with the development-selected set to report leave-one-out conditional gains, and set `FROZEN_PARENT_CHILD_REPLACEMENTS` to confirm parent-to-child-set and individual-child replacements while holding the remaining population fixed. The initial seed catalog is pinned at hash `54d79903b1dbf073`. After the development refinement round, update `EXPECTED_FROZEN_CATALOG_VERSION` to the reviewed final catalog hash before confirmation. The code guard checks the hash; the selected pathogen limit and pair split must match notebooks 10a–10d.

If a definition or selection changes after reviewing confirmation performance, those pairs have become development evidence for that decision. A new independent confirmation set would be needed for a fresh final estimate.

In [ ]:
CONTEXT = load_concept_catalog_context(PROJECT_ROOT)
EXPECTED_FROZEN_CATALOG_VERSION = "54d79903b1dbf073"  # Update only after freezing a different catalog.
CONFIRMATION_ENABLED = True
FROZEN_POPULATION = []  # Copy the development-selected concepts here before confirmation.
FROZEN_PARENT_CHILD_REPLACEMENTS = {}  # e.g. {"surveillance": ["genomic_surveillance", "hospital_surveillance"]}


## Held-out rolling-origin confirmation

Shared forecast slopes are learned using development pairs. For each confirmation pair, its intercept is calibrated from that pair's SI history available at each origin; confirmation outcomes do not train shared slopes. Only confirmation-pair forecasts are kept in the final table.

The fixed lag length, direct horizons, eligibility threshold, and MSE loss match notebook 10d, making the development and confirmation summaries comparable.

## Pair-specific concept ranking (exploratory)

The report ranks concepts for every pathogen–antibiotic pair using development-fitted direct-horizon panel models. Each model adjusts for pair and origin-year intercepts, the previous three SI values, publication volume, and the other frozen concept signals. The pair score is the fitted contribution of its observed concept history; the accompanying 10 percentage-point contrast shifts all three lagged yearly prevalence signals together. Rankings average the three horizon contributions with equal weights.

**Interpretation:** these are assumption-dependent counterfactual attributions, not identified causal effects. A causal reading additionally needs a well-defined intervention, adequate overlap, and no unmeasured time-varying confounding. The available workflow does not measure all relevant AMR burden, policy, and surveillance changes. Check status, support counts, and uncertainty columns; unsupported or non-identifiable estimates are left unranked. Confirmation-pair outcomes are not used to fit the shared coefficients.

In [ ]:
RESULT = confirm_frozen_concepts(
    PROJECT_ROOT, expected_catalog_version=EXPECTED_FROZEN_CATALOG_VERSION,
    enabled=CONFIRMATION_ENABLED, frozen_population=FROZEN_POPULATION,
    parent_child_replacements=FROZEN_PARENT_CHILD_REPLACEMENTS,
    fit_start_year=FIT_START_YEAR, fit_end_year=FORECAST_ORIGIN_END_YEAR,
    lag_length=LAG_LENGTH, horizons=FORECAST_HORIZONS,
    min_prior_observations=MIN_PRIOR_OBSERVATIONS,
    min_attribution_origins=MIN_ATTRIBUTION_ORIGINS,
)
display(RESULT["population_summary"])


## Report and plot

The table reports held-out mean gain and specialist strength separately at each configured horizon and as an equal-weight aggregate. The figure shows mean gain by configured horizon; the zero line marks parity with the reference model. Read it alongside horizon rows and confirmation-pair counts. Keep negative results too: they show where the frozen vocabulary does not generalise.

In [ ]:
display(RESULT["horizon_report"])
ranking = RESULT["pair_concept_causal_ranking"].sort_values(
    ["pathogen", "antibiotic", "causal_rank", "concept_id"], na_position="last"
)
display(ranking)
display(RESULT["figure"])
print("Saved confirmation figure:", RESULT["figure_path"])
print("Saved pair ranking:", RESULT["paths"]["pair_concept_causal_ranking"])
print("Saved horizon estimates:", RESULT["paths"]["pair_concept_causal_horizons"])